# H2O AutoML Pipeline

This notebook walks through a basic H2O AutoML workflow: loading data, splitting it, training multiple models automatically, and inspecting the leaderboard.

## 1. Initialize the H2O Cluster

In [1]:
import h2o
from h2o.automl import H2OAutoML

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Temurin-25.0.2+10 (build 25.0.2+10-LTS, mixed mode, sharing)
  Starting server from C:\Users\User\anaconda3\envs\cmi-flu-prediction-challenge-capstone\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\User\AppData\Local\Temp\tmpnbnznec1
  JVM stdout: C:\Users\User\AppData\Local\Temp\tmpnbnznec1\h2o_User_started_from_python.out
  JVM stderr: C:\Users\User\AppData\Local\Temp\tmpnbnznec1\h2o_User_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,America/Los_Angeles
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,1 month and 3 days
H2O_cluster_name:,H2O_from_python_User_x8flm3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,15.41 Gb
H2O_cluster_total_cores:,16
H2O_cluster_allowed_cores:,16
H2O_cluster_status:,"locked, healthy"


## 2. Import the Data

In [2]:
data = h2o.import_file("merged_data/combined.parquet")

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


## 3. Identify Predictors and Response

In [10]:
x = data.columns
y = "HAI_Vic B/Austria/1359417/2021_28"
tmp = data[y]
x.remove(y)

## 4. Convert Response to Factor (for Classification)

In [4]:
# data[y] = data[y].asfactor()

## 5. Split into Train and Test Sets

In [5]:
train, test = data.split_frame(ratios=[.8], seed=1234)

## 6. Run AutoML (20 Base Models, 1-Hour Cap)

In [6]:
aml = H2OAutoML(max_models=4, seed=1, max_runtime_secs=300)
aml.train(x=x, y=y, training_frame=train)

AutoML progress: |
23:37:04.159: AutoML: XGBoost is not available; skipping it.

██████████
23:37:46.558: _train param, Dropping bad and constant columns: [d0_ENSG00000259312_y, d0_ENSG00000257155_x, d0_ENSG00000257155_y, d7_ENSG00000258986_x, d7_ENSG00000258986_y, d7_ENSG00000289275, d7_ENSG00000300155, d0_ENSG00000259708_x, d7_ENSG00000300156, d7_ENSG00000300157, d7_ENSG00000300151, d7_ENSG00000248378_x, d7_ENSG00000300152, d7_ENSG00000248378_y, d7_ENSG00000289279, d7_ENSG00000300159, d7_ENSG00000289270, d0_ENSG00000229437_x, d0_ENSG00000229437_y, d7_ENSG00000278013_y, d7_ENSG00000289264, d7_ENSG00000300143, d7_ENSG00000289265, d7_ENSG00000300144, d7_ENSG00000289266, d7_ENSG00000300145, d7_ENSG00000289261, d7_ENSG00000300140, d7_ENSG00000300141, d7_ENSG00000300147, d7_ENSG00000300148, d7_ENSG00000253551_y, d7_ENSG00000253551_x, d7_ENSG00000300180, d7_ENSG00000300181, d7_ENSG00000223530_y, d7_ENSG00000300176, d7_ENSG00000278013_x, d7_ENSG00000300177, d7_ENSG00000300178, d7_ENSG0000030

,family,link,regularization,lambda_search,number_of_predictors_total,number_of_active_predictors,number_of_iterations,training_frame
,gaussian,identity,Ridge ( lambda = 35.09 ),"nlambda = 30, lambda.max = 35.09, lambda.min = 35.09, lambda.1se = -1.0",156300,26966,3,AutoML_1_20260415_233558_training_py_2_sid_915b
,timestamp,duration,iteration,lambda,predictors,deviance_train,deviance_test,alpha
,2026-04-15 23:38:54,0.000 sec,3,.35E2,26967,2.1458305,4.7064841,0.0


## 7. View the AutoML Leaderboard

In [7]:
lb = aml.leaderboard
print(lb.head(rows=lb.nrows))

model_id                           rmse      mse      mae     rmsle    mean_residual_deviance
GLM_1_AutoML_1_20260415_233558  2.16944  4.70648  1.75372  0.355982                   4.70648
[1 row x 6 columns]



## 8. Inspect the Best Model

In [8]:
print(aml.leader)

Model Details
H2OGeneralizedLinearEstimator : Generalized Linear Modeling
Model Key: GLM_1_AutoML_1_20260415_233558


GLM Model: summary
    family    link      regularization            lambda_search                                                            number_of_predictors_total    number_of_active_predictors    number_of_iterations    training_frame
--  --------  --------  ------------------------  -----------------------------------------------------------------------  ----------------------------  -----------------------------  ----------------------  -----------------------------------------------
    gaussian  identity  Ridge ( lambda = 35.09 )  nlambda = 30, lambda.max = 35.09, lambda.min = 35.09, lambda.1se = -1.0  156300                        26966                          3                       AutoML_1_20260415_233558_training_py_2_sid_915b

ModelMetricsRegressionGLM: glm
** Reported on train data. **

MSE: 4.441615640162482
RMSE: 2.107514090145658
MAE: 1.78110392429

In [14]:
predictions = aml.leader.predict(test)

results = test[y].cbind(predictions["predict"])
results.columns = ["actual", "predicted"]

# Filter out rows where actual is NaN
results_clean = results[results["actual"].isna() == 0]

print(results_clean.head(20))

glm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
  actual    predicted
 2.32193      2.68947
 3.32193      3.57382
 3.32193      2.42061
 5.32193      3.68805
 2.32193      2.59926
 6.32193      5.64415
 3.32193      3.28615
 2.32193      3.17215
 6.32193      6.48179
 3.32193      7.36293
 7.32193      4.79584
 6.32193      6.43864
 2.32193      2.27601
 5.32193      7.80459
 2.32193      4.01932
 2.32193      3.27129
 4.32193      5.25505
 2.32193      3.18216
 2.32193      2.97288
 6.32193      5.46333
[20 rows x 2 columns]



In [ ]:
# # Save the model to disk
# h2o.save_model(aml.leader, path="./my_model")
#
# # Load it back later
# saved_model = h2o.load_model("./my_model/model_id_here")